In [14]:
%%time
import copernicusmarine
ds = copernicusmarine.open_dataset(
    dataset_id="cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D",
    variables=["CHL", "flags"], 
    minimum_longitude=42, maximum_longitude=80, minimum_latitude=5, maximum_latitude=31,
    start_datetime="2023-01-01", end_datetime="2024-12-31")
ds.CHL.isel(time=1).load()

INFO - 2026-08-06T05:32:48Z - Selected dataset version: "202603"
INFO - 2026-08-06T05:32:48Z - Selected dataset part: "default"


CPU times: user 2.38 s, sys: 1.09 s, total: 3.47 s
Wall time: 14.3 s


<xarray.DataArray 'CHL' (latitude: 624, longitude: 912)> Size: 2MB
array([[       nan,        nan,        nan, ..., 0.5453817 , 0.53464735,
        0.48105383],
       [       nan,        nan,        nan, ..., 0.48314285, 0.47077876,
        0.4588009 ],
       [       nan,        nan,        nan, ..., 0.50086236, 0.4888252 ,
        0.4814038 ],
       ...,
       [       nan,        nan,        nan, ...,        nan,        nan,
               nan],
       [       nan,        nan,        nan, ...,        nan,        nan,
               nan],
       [       nan,        nan,        nan, ...,        nan,        nan,
               nan]], shape=(624, 912), dtype=float32)
Coordinates:
  * latitude   (latitude) float32 2kB 5.021 5.062 5.104 ... 30.9 30.94 30.98
  * longitude  (longitude) float32 4kB 42.02 42.06 42.1 ... 79.9 79.94 79.98
    time       datetime64[ns] 8B 2023-01-02
Attributes:
    units:          milligram m-3
    long_name:      Chlorophyll-a concentration - Mean of the binned pixels
    valid_min:      0.0
    standard_name:  mass_concentration_of_chlorophyll_a_in_sea_water
    valid_max:      1000.0

In [4]:
%%time


CPU times: user 1.58 s, sys: 640 ms, total: 2.22 s
Wall time: 7.86 s


<xarray.DataArray 'CHL' (lat: 624, lon: 912)> Size: 2MB
array([[       nan,        nan,        nan, ..., 0.5453817 , 0.53464735,
        0.48105383],
       [       nan,        nan,        nan, ..., 0.48314285, 0.47077876,
        0.4588009 ],
       [       nan,        nan,        nan, ..., 0.50086236, 0.4888252 ,
        0.4814038 ],
       ...,
       [       nan,        nan,        nan, ...,        nan,        nan,
               nan],
       [       nan,        nan,        nan, ...,        nan,        nan,
               nan],
       [       nan,        nan,        nan, ...,        nan,        nan,
               nan]], shape=(624, 912), dtype=float32)
Coordinates:
  * lat      (lat) float32 2kB 5.021 5.062 5.104 5.146 ... 30.9 30.94 30.98
  * lon      (lon) float32 4kB 42.02 42.06 42.1 42.15 ... 79.85 79.9 79.94 79.98
    time     datetime64[ns] 8B 2023-01-02
Attributes:
    standard_name:  mass_concentration_of_chlorophyll_a_in_sea_water
    long_name:      Chlorophyll-a concentration - Mean of the binned pixels
    valid_max:      1000.0
    units:          milligram m-3
    valid_min:      0.0

In [15]:
%%time
import icechunk as ic
import xarray as xr
import zarr

def open_globcolour(lat_slice, lon_slice):
    """Open a spatial subset as a Dask-backed xarray Dataset."""
    url = "https://data.source.coop/fish-pace/globcolour/cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D"
    storage = ic.http_storage(url)
    repo = ic.Repository.open(storage)
    auth = {p: ic.credentials.HttpAccess for p in repo.config.virtual_chunk_containers or []}
    store = ic.Repository.open(storage, authorize_virtual_chunk_access=auth).readonly_session("main").store

    ds = xr.open_zarr(store, consolidated=False, chunks=None)
    ds = ds.sel(lat=lat_slice, lon=lon_slice)

    root = zarr.open_group(store=store, mode="r")
    chunk_map = dict(zip(ds["CHL"].dims, root["CHL"].chunks))

    return ds.chunk(chunk_map)

ds_sc = open_globcolour(slice(31, 5), slice(42, 80))
ds_sc.CHL.isel(time=1).load()

/srv/conda/envs/notebook/lib/python3.12/site-packages/zarr/codecs/numcodecs/_codecs.py:141: ZarrUserWarning: Numcodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  super().__init__(**codec_config)
/srv/conda/envs/notebook/lib/python3.12/site-packages/zarr/codecs/numcodecs/_codecs.py:141: ZarrUserWarning: Numcodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  super().__init__(**codec_config)


CPU times: user 3.62 s, sys: 1.16 s, total: 4.78 s
Wall time: 5.31 s


<xarray.DataArray 'CHL' (lat: 624, lon: 912)> Size: 2MB
array([[nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       ...,
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan]],
      shape=(624, 912), dtype=float32)
Coordinates:
  * lat      (lat) float32 2kB 30.98 30.94 30.9 30.85 ... 5.104 5.062 5.021
  * lon      (lon) float32 4kB 42.02 42.06 42.1 42.15 ... 79.85 79.9 79.94 79.98
    time     datetime64[ns] 8B 1997-09-06
Attributes:
    input_files_reprocessings:  Processor version: SeaWiFS R2022.0
    type:                       surface
    ancillary_variables:        flags CHL_uncertainty
    standard_name:              mass_concentration_of_chlorophyll_a_in_sea_water
    long_name:                  Chlorophyll-a concentration - Mean of the bin...
    valid_min:                  0.0
    units:                      milligram m-3
    coverage_content_type:      modelResult
    valid_max:                  1000.0

In [16]:
%%time
import icechunk as ic
import xarray as xr
import zarr

def open_globcolour(lat_slice, lon_slice):
    """Open a spatial subset as a Dask-backed xarray Dataset."""
    url = "https://data.source.coop/fish-pace/globcolour/cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D"
    storage = ic.http_storage(url)
    repo = ic.Repository.open(storage)
    auth = {p: ic.credentials.HttpAccess for p in repo.config.virtual_chunk_containers or []}
    store = ic.Repository.open(storage, authorize_virtual_chunk_access=auth).readonly_session("main").store

    ds = xr.open_zarr(store, consolidated=False, chunks={})
    ds = ds.CHL.sel(lat=lat_slice, lon=lon_slice)

    return ds

ds_sc = open_globcolour(slice(31, 5), slice(42, 80))
ds_sc.isel(time=1).load()


CPU times: user 7.47 s, sys: 1.43 s, total: 8.9 s
Wall time: 9.92 s


<xarray.DataArray 'CHL' (lat: 624, lon: 912)> Size: 2MB
array([[nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       ...,
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan]],
      shape=(624, 912), dtype=float32)
Coordinates:
  * lat      (lat) float32 2kB 30.98 30.94 30.9 30.85 ... 5.104 5.062 5.021
  * lon      (lon) float32 4kB 42.02 42.06 42.1 42.15 ... 79.85 79.9 79.94 79.98
    time     datetime64[ns] 8B 1997-09-06
Attributes:
    input_files_reprocessings:  Processor version: SeaWiFS R2022.0
    type:                       surface
    ancillary_variables:        flags CHL_uncertainty
    standard_name:              mass_concentration_of_chlorophyll_a_in_sea_water
    long_name:                  Chlorophyll-a concentration - Mean of the bin...
    valid_min:                  0.0
    units:                      milligram m-3
    coverage_content_type:      modelResult
    valid_max:                  1000.0

In [19]:
%%time
import icechunk as ic
import xarray as xr
import zarr

def open_globcolour(lat_slice, lon_slice):
    """Open a spatial subset as a Dask-backed xarray Dataset."""
    url = "https://data.source.coop/fish-pace/globcolour/cmems_obs-oc_glo_bgc-plankton_my_l3-multi-4km_P1D"
    storage = ic.http_storage(url)
    repo = ic.Repository.open(storage)
    auth = {p: ic.credentials.HttpAccess for p in repo.config.virtual_chunk_containers or []}
    store = ic.Repository.open(storage, authorize_virtual_chunk_access=auth).readonly_session("main").store

    ds = xr.open_zarr(store, consolidated=False, chunks=None)
    ds = ds.CHL.sel(lat=lat_slice, lon=lon_slice)

    return ds

ds_sc = open_globcolour(slice(31, 5), slice(42, 80))
ds_sc.isel(time=1).load()

/srv/conda/envs/notebook/lib/python3.12/site-packages/zarr/codecs/numcodecs/_codecs.py:141: ZarrUserWarning: Numcodecs codecs are not in the Zarr version 3 specification and may not be supported by other zarr implementations.
  super().__init__(**codec_config)


CPU times: user 3.47 s, sys: 868 ms, total: 4.34 s
Wall time: 4.67 s


<xarray.DataArray 'CHL' (lat: 624, lon: 912)> Size: 2MB
array([[nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       ...,
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan]],
      shape=(624, 912), dtype=float32)
Coordinates:
  * lat      (lat) float32 2kB 30.98 30.94 30.9 30.85 ... 5.104 5.062 5.021
  * lon      (lon) float32 4kB 42.02 42.06 42.1 42.15 ... 79.85 79.9 79.94 79.98
    time     datetime64[ns] 8B 1997-09-06
Attributes:
    input_files_reprocessings:  Processor version: SeaWiFS R2022.0
    type:                       surface
    ancillary_variables:        flags CHL_uncertainty
    standard_name:              mass_concentration_of_chlorophyll_a_in_sea_water
    long_name:                  Chlorophyll-a concentration - Mean of the bin...
    valid_min:                  0.0
    units:                      milligram m-3
    coverage_content_type:      modelResult
    valid_max:                  1000.0